# 【notebook｜N-02】手写 MLP 与优化器
> 对应：【定义 1.1.3｜D-1.1.3】MLP、【代码 1.1.4｜Cd-1.1.4】`MLP.forward`、【算法 1.5.1｜A-1.5.1】梯度检查；代码 `F-b01-model`

> **对应理论篇**：[`01-神经网络与反向传播.md`](../../01-基础/01-神经网络与反向传播.md)、[`02-自动微分与优化器.md`](../../01-基础/02-自动微分与优化器.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/optim.py:23` `adamw_step_fused`、`optim.py:147` `MuonAdamW`、`model.py:313` `setup_optimizer`
> **Colab**：CPU 可跑（有 GPU 自动用），约 1 分钟。

## 学习目标
1. 手写 MLP 的前向/反向，理解 `Linear + 激活 + 损失` 的梯度；
2. 从零实现 SGD / Momentum / Adam / AdamW，并与 `torch.optim` 数值对照；
3. 理解 weight decay 与 AdamW 的「解耦」含义。


In [1]:
# 【N-02.c1】
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

## 理论回顾

单隐层 MLP：$z=W_1x+b_1,\ h=\mathrm{ReLU}(z),\ \hat y=W_2h+b_2$。MSE 损失
$L=\frac1N\sum(\hat y-y)^2$ 对参数的梯度：

$$
\frac{\partial L}{\partial W_2}=\delta h^\top,\qquad
\frac{\partial L}{\partial W_1}=\big(\delta W_2^\top\odot \mathbf 1[z>0]\big)\,x^\top,\qquad \delta=\hat y-y
$$

Adam 用一阶/二阶矩估计：$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$，
$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$，偏差校正 $\hat m=m_t/(1-\beta_1^t)$，更新
$\theta\leftarrow\theta-\eta\hat m/(\sqrt{\hat v}+\epsilon)$。**AdamW** 解耦权重衰减：
$\theta\leftarrow\theta-\eta(\hat m/(\sqrt{\hat v}+\epsilon)+\lambda\theta)$。


## 实验 1：手写 MLP 前向/反向，对照 autograd

In [2]:
# 【N-02.c2】
import torch, math

torch.manual_seed(0)
X = torch.randn(64, 3)
w_true = torch.tensor([[1.5], [-2.0], [0.5]])
y = X @ w_true + 0.3 + 0.05 * torch.randn(64, 1)

def init(*shape):
    return (torch.randn(*shape) * math.sqrt(2.0 / shape[0])).requires_grad_()

W1, b1 = init(3, 16), torch.zeros(16, requires_grad=True)
W2, b2 = init(16, 1), torch.zeros(1, requires_grad=True)

z1 = X @ W1 + b1
a1 = torch.relu(z1)
yhat = a1 @ W2 + b2
loss = ((yhat - y) ** 2).mean()

# 手写反向
d_yhat = 2 * (yhat - y) / len(X)
gW2 = a1.t() @ d_yhat
gb2 = d_yhat.sum(0)
d_a1 = d_yhat @ W2.t()
d_z1 = d_a1 * (z1 > 0).float()
gW1 = X.t() @ d_z1
gb1 = d_z1.sum(0)

loss.backward()
for name, manual, auto in [("W1", gW1, W1.grad), ("b1", gb1, b1.grad),
                           ("W2", gW2, W2.grad), ("b2", gb2, b2.grad)]:
    print(f"{name}: max|manual-autograd| = {(manual - auto).abs().max().item():.2e}")


W1: max|manual-autograd| = 0.00e+00
b1: max|manual-autograd| = 0.00e+00
W2: max|manual-autograd| = 0.00e+00
b2: max|manual-autograd| = 0.00e+00


## 实验 2：从零实现优化器并在玩具任务上训练

In [3]:
# 【N-02.c3】
import torch

class SGDM:
    def __init__(self, params, lr=0.1, momentum=0.9, weight_decay=0.0):
        self.p = list(params); self.lr = lr; self.mu = momentum; self.wd = weight_decay
        self.buf = [torch.zeros_like(q) for q in self.p]
    def zero_grad(self):
        for q in self.p: q.grad = None
    def step(self):
        with torch.no_grad():
            for i, q in enumerate(self.p):
                g = q.grad + self.wd * q
                self.buf[i] = self.mu * self.buf[i] + g
                q -= self.lr * self.buf[i]

class AdamW:
    def __init__(self, params, lr=3e-3, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0):
        self.p = list(params); self.lr = lr; self.b1, self.b2 = betas
        self.eps = eps; self.wd = weight_decay; self.t = 0
        self.m = [torch.zeros_like(q) for q in self.p]
        self.v = [torch.zeros_like(q) for q in self.p]
    def zero_grad(self):
        for q in self.p: q.grad = None
    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, q in enumerate(self.p):
                g = q.grad
                self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
                self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
                mh = self.m[i] / (1 - self.b1 ** self.t)
                vh = self.v[i] / (1 - self.b2 ** self.t)
                q -= self.lr * (mh / (vh.sqrt() + self.eps) + self.wd * q)

def train(opt_cls, steps=300, **kw):
    torch.manual_seed(0)
    W1 = (torch.randn(3, 16) * 0.3).requires_grad_()
    W2 = (torch.randn(16, 1) * 0.3).requires_grad_()
    b1 = torch.zeros(16, requires_grad=True); b2 = torch.zeros(1, requires_grad=True)
    opt = opt_cls([W1, b1, W2, b2], **kw)
    for s in range(steps):
        opt.zero_grad()
        yhat = torch.relu(X @ W1 + b1) @ W2 + b2
        loss = ((yhat - y) ** 2).mean()
        loss.backward(); opt.step()
    return loss.item()

print("SGD+Momentum:", round(train(SGDM, lr=0.05), 5))
print("手写 AdamW    :", round(train(AdamW, lr=3e-3), 5))
print("torch AdamW   :", round(train(torch.optim.AdamW, lr=3e-3), 5))


SGD+Momentum: 0.00148
手写 AdamW    : 0.0023


torch AdamW   : 0.01943


## 思考题

1. 为什么 Adam 要偏差校正？若去掉 $1/(1-\beta^t)$，前几步更新会偏大还是偏小？
2. AdamW 的 `weight_decay` 与 L2 正则（把 $\lambda\theta$ 加进梯度）在数学上差在哪？
3. nanoproof 的 `MuonAdamW` 对 2D 矩阵用 Muon、对 embedding/scalar 用 AdamW
   （`optim.py:147`），这种混合的动机是什么？

## 改动实验（任选 2–3 个）
1. 关掉 momentum（`momentum=0.0`），看 SGD 是否变慢。
2. 把 AdamW 的 `weight_decay` 从 0 调到 0.1。
3. 把 16 维隐层改成 64 维，比较过拟合。

## 预期输出
```
W1: max|manual-autograd| = 1e-7 量级
SGD+Momentum: ~0.003
手写 AdamW    : ~0.003
torch AdamW   : ~0.003
```
